This notebook shows how to form a simple pipeline of retrieval augmented generation (RAG) using Langchain and three models, to retrieve information from PDF files and answer questions related to the file. 

The models including a embedding model, a reranking model, and a large language model to make a simple RAG, which can retrieve answers from the file based on the questions we ask.

You don't need to download the models, as the models are hosted at NVIDIA endpoints (https://build.nvidia.com/). You need to generate a NVIDIA API key to use the model endpoints.


In [1]:
!pip install --upgrade --quiet pip arxiv pymupdf  pypdf langchain langchain-nvidia-ai-endpoints  langchain-community faiss-gpu faiss-cpu

You can choose any PDF file to learn more information, Langchain provides PyPDFLoader for PDF files:

In [2]:
from langchain_community.document_loaders import PyPDFLoader
# You can replace the below link with a different link to a PDF file 
#An example of NVIDIA financial report 2025 PDF
loader = PyPDFLoader("https://d18rn0p25nwr6d.cloudfront.net/CIK-0001045810/177440d5-3b32-4185-8cc8-95500a9dc783.pdf")

pages = loader.load_and_split()
pages[0]

Document(metadata={'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2025-02-26T16:49:22-05:00', 'title': '0001045810-25-000023', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2025-02-26 for the period ending 2025-01-26', 'keywords': '0001045810-25-000023; ; 10-K', 'moddate': '2025-02-26T16:49:44-05:00', 'source': 'https://d18rn0p25nwr6d.cloudfront.net/CIK-0001045810/177440d5-3b32-4185-8cc8-95500a9dc783.pdf', 'total_pages': 130, 'page': 0, 'page_label': '1'}, page_content='Table of Contents\nUNITED STATESSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\n____________________________________________________________________________________________\nFORM 10-K\n☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934\n    For the fiscal year ended January 26, 2025\nOR\n☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCH

A large language model is required to answer questions later on.

In [3]:
 # Initialize LLM
import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# NVIDIA AI Foundation Endpoints
#os.environ["NVIDIA_API_KEY"] = "your API key starting with nvapi-" 
os.environ["NVIDIA_API_KEY"] = "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"
llm = ChatNVIDIA(
  model="meta/llama-3.1-8b-instruct",
  temperature=0.2,
  top_p=0.7,
  max_tokens=1024,
)

A simple example of embedding model: choose any sentence from above file to see its embedding:

In [4]:
# You can replace the example text with any text you want to try.
example_text = "We have invested over $58.2 billion in research and development since our inception \n"

In [5]:
# Embedding  
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
embedding_model = NVIDIAEmbeddings(model="nvidia/llama-3.2-nv-embedqa-1b-v2", truncate="END")
embedding_model.embed_query(example_text)

[-0.035003662109375,
 -0.0051727294921875,
 -0.0012226104736328125,
 0.0308074951171875,
 0.0305633544921875,
 0.00640106201171875,
 -0.02838134765625,
 -0.00039649009704589844,
 0.01666259765625,
 0.0008139610290527344,
 -0.0185089111328125,
 -0.019073486328125,
 -0.003971099853515625,
 -0.007663726806640625,
 0.02001953125,
 -0.0223541259765625,
 0.020477294921875,
 0.044921875,
 0.0028171539306640625,
 0.048370361328125,
 -0.0182037353515625,
 0.00818634033203125,
 0.007472991943359375,
 0.05999755859375,
 0.0221710205078125,
 -0.004016876220703125,
 -0.0322265625,
 0.004741668701171875,
 0.0238494873046875,
 0.004913330078125,
 -0.03338623046875,
 -0.0091552734375,
 -0.0198211669921875,
 -0.0115509033203125,
 0.0250396728515625,
 0.028961181640625,
 -0.0231475830078125,
 -0.01090240478515625,
 0.0218963623046875,
 -0.040435791015625,
 0.0008397102355957031,
 -0.022125244140625,
 0.0162811279296875,
 0.005474090576171875,
 0.0214080810546875,
 -0.0009732246398925781,
 -0.03125,
 -0.

Let's see an example of adding the NVIDIA financial report into the knowledge base so that questions requires the information of the report can be answered.

Before adding the report into the knowledge base, the large language model cannot give an answer to a specific question related to the file, such as "How much has NVIDIA invested in research and development since their inception?”

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a helpful and friendly AI!"
        "Your responses should be concise and no longer than two sentences."
        "Do not hallucinate. Say you don't know if you don't have this information."
    )),
    ("user", "{question}")
])

chain = prompt | llm | StrOutputParser()

In [7]:
# You can replace this question with any question that is specific to the research paper you plan to learn more
print(chain.invoke("How much has NVIDIA invested in research and development since their inception?"))

I don't have specific information on NVIDIA's total research and development investment since their inception. However, NVIDIA has consistently reported significant R&D expenses in their annual financial reports, with a notable increase in recent years.


The answer is not correct. This is because the answer was generated by the large language model which was not trained with the knowledge of this report. Although the large language model can handle general questions pretty well, such as:

In [8]:
print(chain.invoke({"question": "What is a large language model"}))

A large language model is a type of artificial intelligence (AI) that uses complex algorithms and large datasets to process and generate human-like language. It's a computer program that can understand, interpret, and respond to natural language inputs, often with a high degree of accuracy and fluency.


To answer the specific question related to this financial report, we can create a knowledge base by adding the report: 

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", ";", ",", " ", ""],
)

chunks = text_splitter.split_documents(pages)
len(chunks)

576

In [10]:
from langchain.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embedding=embedding_model)

In [11]:
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_messages([
    ("system", 
        "You are a helpful and friendly AI!"
        "Your responses should be concise and no longer than two sentences."
        "Do not hallucinate. Say you don't know if you don't have this information."
        # "Answer the question using only the context"
        "\n\nQuestion: {question}\n\nContext: {context}"
    ),
    ("user", "{question}")
])

chain = (
    {
        'context': vector_store.as_retriever(),
        'question': (lambda x:x)
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [12]:
# You can replace this question with any question that is specific to the research paper you plan to learn more
print(chain.invoke("How much has NVIDIA invested in research and development since their inception?"))

NVIDIA has invested over $58.2 billion in research and development since their inception.


Now we have the correct answer which means the LLM was able to retrieve the information of the financial report that was added into the knowledge base. Let's try to ask more questions about the report: 

In [13]:
# You can replace this question with any question that is specific to the research paper you plan to learn more
print(chain.invoke("How many developers are there using CUDA worldwide?"))
#Answer in the PDF file: There are over 5.9 million developers worldwide using CUDA and our other software tools to help deploy our technology in our target markets

There are over 5.9 million developers worldwide using CUDA.


In [14]:
# You can replace this question with any question that is specific to the research paper you plan to learn more
print(chain.invoke("What are the key factors that drove NVIDIA revenue growth in fiscal year 2025?"))
#Answer in the PDF file: Revenue growth in fiscal year 2025 was driven by data center compute and networking platforms for accelerated computing and AI solutions

The key factors that drove NVIDIA revenue growth in fiscal year 2025 were strong demand for its accelerated computing and AI solutions, particularly its Hopper architecture used for large language models, recommendation engines, and generative AI applications, as well as sales of its GeForce RTX 40 Series GPUs.


Finally, let's see how reranking model works: 

In [15]:
# Retrieve K relevant results to the query
query = "HWhat are the key factors that drove NVIDIA revenue growth in fiscal year 2025?"
retrieved_docs = vector_store.similarity_search(query, k=5, fetch_k=5)
len(retrieved_docs)
#for doc in retrieved_docs:
#    print(doc.page_content)
#    print(doc.metadata)

5

After retrieving K results to the query, let reranker calculate the similarity scores, and pick the answer with the highest score:

In [16]:
from langchain_nvidia_ai_endpoints import NVIDIARerank
from langchain_core.documents import Document

query = "What are the key factors that drove NVIDIA revenue growth in fiscal year 2025?"

passages = [docs.page_content for docs in retrieved_docs]

client = NVIDIARerank(
  model="nvidia/llama-3.2-nv-rerankqa-1b-v2", top_n=3
)

response = client.compress_documents(
  query=query,
  documents=[Document(page_content=passage) for passage in passages]   
)

response

[Document(metadata={'relevance_score': 17.0625}, page_content='applications.\nOur two operating segments are "Compute & Networking" and "Graphics." Refer to Note 16 of the Notes to the Consolidated Financial Statements in Part IV,Item 15 of this Annual Report on Form 10-K for additional information.\nHeadquartered in Santa Clara, California, NVIDIA was incorporated in California in April 1993 and reincorporated in Delaware in April 1998.\nRecent Developments, Future Objectives and Challenges\nDemand and Supply\nRevenue growth in fiscal year 2025 was driven by data center compute and networking platforms for accelerated computing and AI solutions. Demand for ourHopper architecture drove our significant growth for the full year. We began shipping production systems of the Blackwell architecture in the fourth quarter of\nfiscal year 2025.\nDemand estimates for our products, applications, and services can be incorrect and create volatility in our revenue or supply levels. We may not be abl

In [17]:
print(f"Most relevant: {response[0].page_content}\n")

Most relevant: applications.
Our two operating segments are "Compute & Networking" and "Graphics." Refer to Note 16 of the Notes to the Consolidated Financial Statements in Part IV,Item 15 of this Annual Report on Form 10-K for additional information.
Headquartered in Santa Clara, California, NVIDIA was incorporated in California in April 1993 and reincorporated in Delaware in April 1998.
Recent Developments, Future Objectives and Challenges
Demand and Supply
Revenue growth in fiscal year 2025 was driven by data center compute and networking platforms for accelerated computing and AI solutions. Demand for ourHopper architecture drove our significant growth for the full year. We began shipping production systems of the Blackwell architecture in the fourth quarter of
fiscal year 2025.
Demand estimates for our products, applications, and services can be incorrect and create volatility in our revenue or supply levels. We may not be able to
generate significant revenue from them. Advancemen

In [18]:
print(f"Least relevant: {response[-1].page_content}\n")

Least relevant: Professional Visualization revenue for fiscal year 2025 was up 21% from a year ago, driven by the continued ramp of Ada RTX GPU workstations for use casessuch as generative AI-powered design, simulation, and engineering.
Automotive revenue for fiscal year 2025 was up 55% from a year ago, driven by sales of our self-driving platforms.
Gross margin increased in fiscal year 2025 driven by a higher mix of Data Center revenue.
Operating expenses for fiscal year 2025 were up 45% from a year ago, driven by higher compensation and benefits expenses due to employee growth and
compensation increases, and engineering development, compute and infrastructure costs for new product introductions.
Critical Accounting Estimates
Our consolidated financial statements are prepared in accordance with accounting principles generally accepted in the United States, or U.S. GAAP. The



## Reference:

NVIDIA: https://build.nvidia.com/explore/discover

Langchain: https://python.langchain.com/